# Optimización y series temporales

Votación ponderada de clasificadores clásicos y la tarea «caótica 20D» (reglas enredadas en 20 dimensiones).

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

> **Medido en el banco** (`laboratorio/banco/`): la votación ponderada está en `--clasicos` y la tarea caótica es `caotica_20d`. Los generadores de series sintéticas que había aquí están en `laboratorio/series.py`. Ver la sección «cuaderno 08» de `laboratorio/banco/README.md`.

## Votación ponderada (weighted_vote)

5 clasificadores votan con sus probabilidades, cada uno pesado por su exactitud en validación cruzada.

In [ ]:
import numpy as np
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# --- Datos sintéticos ---
X, y = make_classification(n_samples=600, n_features=2, n_informative=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42, class_sep=1.2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- Modelos base ---
base_classifiers = {
    "LogisticRegression": LogisticRegression(random_state=42),
    "KNeighbors": KNeighborsClassifier(n_neighbors=7),
    # SVC(probability=True) queda obsoleto en scikit-learn 1.11: las probabilidades se calibran aparte.
    "SVC_Linear": CalibratedClassifierCV(SVC(kernel='linear', random_state=42), ensemble=False),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "GaussianNB": GaussianNB()
}

# --- Validación cruzada para obtener pesos ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = {name: cross_val_score(clf, X_train, y_train, cv=kf, scoring='accuracy').mean()
          for name, clf in base_classifiers.items()}
weights = {k: v / sum(scores.values()) for k, v in scores.items()}

# --- Reentrenamiento completo ---
trained_models = {}
for name, clf in base_classifiers.items():
    clf.fit(X_train, y_train)
    trained_models[name] = clf

# --- Guardar modelos y pesos ---
joblib.dump(trained_models, '/content/ensemble_models.pkl')
joblib.dump(weights, '/content/ensemble_weights.pkl')
joblib.dump((X_test, y_test), '/content/test_data.pkl')

print("✅ Modelos y pesos guardados.")

In [ ]:
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

# --- Cargar modelos, pesos y datos ---
# Seguridad: joblib/pickle puede ejecutar código; carga solo archivos que hayas creado tú.
models = joblib.load('/content/ensemble_models.pkl')
weights = joblib.load('/content/ensemble_weights.pkl')
X_test, y_test = joblib.load('/content/test_data.pkl')

def weighted_vote(X, models, weights, num_classes=2):
    proba_total = np.zeros((X.shape[0], num_classes))
    for name, model in models.items():
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            proba_total += proba * weights[name]
    return np.argmax(proba_total, axis=1)

# --- Evaluación ---
y_pred = weighted_vote(X_test, models, weights)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"🎯 Accuracy ensamblado: {acc:.4f} | F1-Weighted: {f1:.4f}")

# --- Frontera de decisión (visual opcional) ---
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))

Z = weighted_vote(np.c_[xx.ravel(), yy.ravel()], models, weights)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=ListedColormap(['#FFAAAA', '#AAFFAA']), alpha=0.6)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=ListedColormap(['#FF0000', '#00FF00']), edgecolor='k')
plt.title("Fronteras de Decisión del Ensamblado")
plt.show()

### Stacking: aprender a combinar los votos

Recuperado de tu cuaderno completo (celda 397). En vez de pesar cada clasificador por su exactitud, una regresión logística aprende a combinar sus probabilidades (entrenada con validación cruzada para no hacer trampa). Medido en el banco con `--clasicos` como «Stacking (cuaderno completo)».

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier # Un modelo base adicional
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV  # SVC(probability=True) queda obsoleto en scikit-learn 1.11
from sklearn.metrics import accuracy_score, classification_report
import warnings

# Suprimir advertencias para una salida más limpia
warnings.filterwarnings("ignore")

# --- 1. Generar un dataset sintético ---
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           n_redundant=0, n_clusters_per_class=1,
                           random_state=42, class_sep=1.2) # Un dataset un poco desafiante

# Dividir el dataset en conjuntos de entrenamiento y prueba
# Usamos un conjunto de test final para evaluar el rendimiento general del stack
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset completo: {X.shape[0]} muestras, {X.shape[1]} características")
print(f"Conjunto de entrenamiento para stacking: {X_train_full.shape[0]} muestras")
print(f"Conjunto de prueba final: {X_test_final.shape[0]} muestras")

# --- 2. Definir los Modelos Base (Clasificadores de Nivel 0) ---
# Elegimos una variedad de modelos con diferentes fortalezas
base_models = [
    ('log_reg', LogisticRegression(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=7)), # Ajusta n_neighbors según necesidad
    ('dtree', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)), # Random Forest como un ensamblado
    ('svc', CalibratedClassifierCV(SVC(kernel='linear', random_state=42), ensemble=False)) # SVC con probabilidades
]

# --- 3. Definir el Meta-Modelo (Clasificador de Nivel 1) ---
# Un modelo simple suele funcionar bien como meta-modelo
meta_model = LogisticRegression(random_state=42)

# --- 4. Entrenamiento de los Modelos Base y Generación de Predicciones para el Meta-Modelo ---
# Esto se hace usando Validación Cruzada (CV) para evitar el "data leakage" (fuga de datos).
# Cada modelo base hace predicciones sobre las folds de validación, y estas se usan para entrenar el meta-modelo.

NFOLDS = 5 # Número de folds para la validación cruzada
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

# Almacenaremos las predicciones de los modelos base para entrenar el meta-modelo
# Cada columna será la predicción de un modelo base
# Para problemas binarios, a menudo se usan las probabilidades (predict_proba)
meta_features_train = np.zeros((X_train_full.shape[0], len(base_models)))
meta_features_test = np.zeros((X_test_final.shape[0], len(base_models)))

print("\n--- Entrenando Modelos Base y Recopilando Predicciones para el Meta-Modelo ---")

for i, (name, model) in enumerate(base_models):
    print(f"  Entrenando y recopilando para: {name}")

    # Almacena las predicciones para el conjunto de prueba (X_test_final)
    # Se entrena el modelo base sobre el X_train_full y se predice sobre X_test_final
    # Luego se promedian esas predicciones a través de las folds.
    test_preds_fold = np.zeros((X_test_final.shape[0], NFOLDS))

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_full, y_train_full)):
        X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
        y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]

        # Entrenar el modelo base en el fold de entrenamiento
        model.fit(X_train_fold, y_train_fold)

        # Generar predicciones (probabilidades de la clase 1) para el fold de validación
        # Esto será la entrada para el meta-modelo
        meta_features_train[val_idx, i] = model.predict_proba(X_val_fold)[:, 1]

        # Generar predicciones (probabilidades de la clase 1) para el conjunto de prueba final
        # Estas se promediarán más tarde para formar las características de prueba para el meta-modelo
        test_preds_fold[:, fold_idx] = model.predict_proba(X_test_final)[:, 1]

    # Calcular el promedio de las predicciones en el conjunto de prueba final
    meta_features_test[:, i] = np.mean(test_preds_fold, axis=1)

# --- 5. Entrenar el Meta-Modelo ---
print("\n--- Entrenando el Meta-Modelo ---")
meta_model.fit(meta_features_train, y_train_full)

# --- 6. Hacer Predicciones Finales con el Meta-Modelo ---
print("\n--- Haciendo Predicciones Finales con el Stacking Ensemble ---")
final_predictions = meta_model.predict(meta_features_test)
final_probabilities = meta_model.predict_proba(meta_features_test)[:, 1]

# --- 7. Evaluar el Rendimiento del Stacking Ensemble ---
print("\n--- Evaluación del Stacking Ensemble ---")
print(classification_report(y_test_final, final_predictions))
print(f"Accuracy del Stacking Ensemble: {accuracy_score(y_test_final, final_predictions):.4f}")

# --- Opcional: Comparar con el mejor modelo base individual ---
print("\n--- Comparación con Modelos Base Individuales ---")
best_base_accuracy = 0
best_base_model_name = ""

for name, model in base_models:
    model.fit(X_train_full, y_train_full) # Entrenar el modelo base en todo el X_train_full
    y_pred_base = model.predict(X_test_final)
    acc = accuracy_score(y_test_final, y_pred_base)
    print(f"  Accuracy de {name}: {acc:.4f}")
    if acc > best_base_accuracy:
        best_base_accuracy = acc
        best_base_model_name = name

print(f"\nEl mejor modelo base individual fue '{best_base_model_name}' con Accuracy: {best_base_accuracy:.4f}")
print(f"Accuracy del Stacking Ensemble: {accuracy_score(y_test_final, final_predictions):.4f}")

## Tarea caótica 20D (Chaotic20DClassifier)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = {
    "lr": 5e-4,
    "batch_size": 512,
    "epochs": 1000,
    "early_stop_patience": 50,
    "model_path": "chaotic_best_model.pt",
}

def make_chaotic_20D_data(n=20000, noise=0.8):
    X = np.random.rand(n, 20) * 10 - 5
    y = np.zeros(n, dtype=int)
    for i, p in enumerate(X):
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (abs(p[9]) < .5 and p[12] > 2 and p[18] < -2): y[i] = 0
        elif (p[2] > 1 and p[4] < -.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5): y[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1): y[i] = 2
        else: y[i] = 3
        p[0] += np.random.exponential(noise * 0.1) * np.random.choice([-1, 1])
        p[1] += np.random.normal(0, noise * 0.05)
        p[5] += np.random.poisson(noise * 0.5) * np.random.choice([-1, 1])
        p[19] += np.random.uniform(-noise, noise)
    return X, y

class Chaotic20DClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.01), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.01), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.01),
            nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.net(x)

def load_data(batch_size):
    X, y = make_chaotic_20D_data()
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    val_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader

def train_loop(model, optimizer, criterion, train_loader, val_loader):
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    best_loss = float('inf')
    patience = 0

    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with autocast(enabled=device.type=='cuda'):
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=total_loss/(pbar.n+1))

        val_loss = 0
        model.eval()
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                with autocast(enabled=device.type=='cuda'):
                    out = model(xb)
                    loss = criterion(out, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), config["model_path"])
            patience = 0
        else:
            patience += 1
            if patience >= config["early_stop_patience"]:
                print(f"Early stopping en epoch {epoch+1}")
                break

        print(f"Epoch {epoch+1}: val_loss={val_loss:.4f}, best_loss={best_loss:.4f}")

if __name__ == "__main__":
    train_loader, val_loader = load_data(config["batch_size"])
    model = Chaotic20DClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    train_loop(model, optimizer, criterion, train_loader, val_loader)

In [ ]:
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import os


config = {"model_path": "chaotic_best_model.pt"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.exists(config["model_path"]):
    raise FileNotFoundError(f"Modelo no encontrado en {config['model_path']}")

X, y = make_chaotic_20D_data()
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.long).to(device)

model = Chaotic20DClassifier().to(device)
model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
model.eval()

with torch.no_grad():
    # You can use autocast context manager for mixed precision training on CUDA
    # It's generally good practice to keep it enabled when available
    if device.type == "cuda":
        from torch.cuda.amp import autocast
        with autocast():
            preds = model(X).argmax(dim=1)
    else:
        preds = model(X).argmax(dim=1)


print(classification_report(y.cpu(), preds.cpu()))